# Session 01 · Introduction to MongoDB

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Example 1 - connect, insert, find

In [2]:
from mongo_conn import backend, get_db
print("Backend:", backend())
db = get_db("campus_intro")
db.students.drop(); db.payments.drop()
res = db.students.insert_one({"name": "Ali Raza", "city": "Karachi", "marks": {"python": 78}})
print("inserted _id:", res.inserted_id)
print(db.students.find_one({"name": "Ali Raza"}))

Backend: in-memory mongomock
inserted _id: 6ae01dd589e9c50cb72f2b2f
{'name': 'Ali Raza', 'city': 'Karachi', 'marks': {'python': 78}, '_id': ObjectId('6ae01dd589e9c50cb72f2b2f')}


## Example 2 - flexible schema

In [3]:
db.students.insert_many([
    {"name": "Sana Khan", "city": "Lahore", "skills": ["python", "sql"]},
    {"name": "Hamza Ali", "phone": "0300-1234567"},
])
for d in db.students.find({}, {"_id": 0}):
    print(d)

{'name': 'Ali Raza', 'city': 'Karachi', 'marks': {'python': 78}}
{'name': 'Sana Khan', 'city': 'Lahore', 'skills': ['python', 'sql']}
{'name': 'Hamza Ali', 'phone': '0300-1234567'}


## Example 3 - what's inside an ObjectId

In [4]:
oid = res.inserted_id
print(type(oid).__name__, str(oid), "| created:", oid.generation_time)

ObjectId 6ae01dd589e9c50cb72f2b2f | created: 2026-10-27 03:43:17+00:00


## Example 4 - BSON types from Python

In [5]:
from datetime import datetime
from bson.decimal128 import Decimal128
db.payments.insert_one({"amount": Decimal128("15000.50"), "paid_on": datetime(2026, 10, 27), "late": False})
p = db.payments.find_one({}, {"_id": 0})
print({k: (type(v).__name__, str(v)) for k, v in p.items()})

{'amount': ('Decimal128', '15000.50'), 'paid_on': ('datetime', '2026-10-27 00:00:00'), 'late': ('bool', 'False')}


## Example 5 - count, distinct, update, delete

In [6]:
print("Karachi:", db.students.count_documents({"city": "Karachi"}))
print("cities:", db.students.distinct("city"))
db.students.update_one({"name": "Hamza Ali"}, {"$set": {"city": "Karachi"}})
print("Karachi after update:", db.students.count_documents({"city": "Karachi"}))
db.students.delete_many({"city": {"$exists": False}})
print("total now:", db.students.count_documents({}))

Karachi: 1
cities: ['Karachi', 'Lahore']
Karachi after update: 2
total now: 3


## Example 6 - SQL vs MongoDB

In [7]:
# SELECT name FROM students WHERE city = 'Karachi' ORDER BY name;
print(list(db.students.find({"city": "Karachi"}, {"_id": 0, "name": 1}).sort("name", 1)))

[{'name': 'Ali Raza'}, {'name': 'Hamza Ali'}]
